# Real video: Qwen2.5-VL on NExT-QA frames

End-to-end **video** measurement with the vmd harness: real sampled frames from NExT-QA clips, a real video VLM (Qwen2.5-VL-3B), and frame-sequence corruptions.

What it produces:
- **Collapse gap** — accuracy with frames vs. *without* frames (language prior).
- **Temporal-order curve** — frames shuffled (content preserved, order destroyed).
- **Content curve** — frames dropped (content destroyed).

**Runtime**: GPU L4 recomendada (T4 también vale). ~10 min de descarga (videos.zip 6.5 GB, una vez) + ~25–35 min de inferencia con `N = 60`. Baja `N` para una prueba rápida. Sin entrenamiento.

> ⚠️ Celdas en orden, idempotentes. Si algo queda raro: *Entorno de ejecución → Reiniciar* y ejecutar desde arriba.

In [ ]:
# Setup idempotente
import os, shutil, sys
REPO = "/content/video-modality-diagnostics"
os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q https://github.com/mlahozy21/video-modality-diagnostics.git
os.chdir(REPO)
!pip install -q -e ".[video]"
for k in list(sys.modules):
    if k.startswith("vmd"):
        del sys.modules[k]
sys.path.insert(0, f"{REPO}/src")
import torch
print("setup OK —", os.getcwd(), "| CUDA:", torch.cuda.is_available())

In [ ]:
# Subconjunto real de NExT-QA (anotaciones + solo los vídeos necesarios)
N = 60   # preguntas; baja a 20 para una prueba rápida
!python scripts/prepare_nextqa.py --n {N} --out-dir data/nextqa

In [ ]:
import json

from vmd.config import DiagConfig
from vmd.data import load_jsonl
from vmd.diagnose import run_diagnostic, _eval
from vmd.report import format_report
from vmd.video import VLMVideoBackend

items = load_jsonl("data/nextqa/videoqa.jsonl")
print(f"{len(items)} items de NExT-QA con vídeo real")

# Solo el canal de visión existe en este subset -> el diagnóstico se reduce a
# collapse gap + curvas de robustez (que es justo lo que queremos medir).
config = DiagConfig(modalities=["vision"], severities=[0.0, 0.5, 1.0], seed=0)
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"

## Diagnóstico principal (corrupción = shuffle temporal)

In [ ]:
backend = VLMVideoBackend(MODEL, n_frames=8, corruption_kind="shuffle")
result = run_diagnostic(backend, items, config)
print(format_report(result, backend.name))
json.dump(result, open("results_video_shuffle.json", "w"), indent=1)

## Curva de contenido (corrupción = drop de frames)

Reutiliza el mismo modelo ya cargado (mismo objeto interno) para no duplicar memoria.

In [ ]:
backend_drop = VLMVideoBackend(MODEL, n_frames=8, corruption_kind="drop")
backend_drop._m, backend_drop._proc = backend._m, backend._proc  # comparte pesos

drop_curve = {}
for s in [0.5, 0.75]:
    drop_curve[s] = _eval(backend_drop, items, ["vision"],
                          corrupt={"vision": s}, seed=0)
    print(f"drop severity {s}: accuracy {drop_curve[s]:.3f}")
json.dump(drop_curve, open("results_video_drop.json", "w"), indent=1)

## Tabla para el README

In [ ]:
r = result
rows = [
    ("Full accuracy (8 frames)", r["acc_full"]),
    ("Blind (no frames)", r["blind_language_prior"]),
    ("**Collapse gap**", r["collapse_gap"]),
    ("Shuffle 50% of frames", r["vision_robustness"]["0.5"]),
    ("Shuffle 100% of frames", r["vision_robustness"]["1.0"]),
    ("Drop 50% of frames", drop_curve[0.5]),
    ("Drop 75% of frames", drop_curve[0.75]),
]
print(f"| Metric (Qwen2.5-VL-3B, NExT-QA n={r['n']}) | Accuracy |")
print("|---|---:|")
for name, v in rows:
    print(f"| {name} | {v:.3f} |")

## Cómo leerlo

- **Collapse gap grande** → el VLM de verdad mira los frames en NExT-QA; pequeño → responde desde el prior lingüístico (fenómeno documentado en la literatura de VideoQA).
- **Shuffle plano + drop cayendo** → usa el *contenido* de los frames pero no su *orden temporal* (colapso temporal: típico en preguntas causales/temporales).
- **Ambas curvas planas** → ni siquiera necesita el contenido: colapso total al prior.

Pega la tabla en el README bajo *Real video* (etiquetada con modelo y n), y descarga `results_video_*.json` antes de cerrar el runtime.